In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, recall_score, roc_auc_score
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
import pandas as pd
import numpy as np

In [2]:
def categorize_conflict(x, q1, q3):
    if x == 0:
        return 0
    elif x <= q1:
        return 1
    elif x <= q3:
        return 2
    else:
        return 3

In [3]:
parquet_path = '../data/output/grid_conflict_climate_2019_23.parquet'

#Drop nas #
df = pd.read_parquet(parquet_path)
df = df.dropna()
df['target'] = (df['conflict_count'] >= 1).astype(int)
features = df.drop(['GEOID', 'conflict_count', 'target'], axis=1)
features = pd.get_dummies(features, columns=['year'], prefix='year')
X = features
y = df['target']

In [4]:
parquet_path = '../data/output/grid_conflict_climate_2019_23.parquet'

#Drop nas #
df = pd.read_parquet(parquet_path)
df = df.dropna()
new_df = df[df["conflict_count"] > 0]
q1, q2, q3 = new_df['conflict_count'].quantile(0.25), new_df['conflict_count'].quantile(0.50), new_df['conflict_count'].quantile(0.75)

df['target'] = df['conflict_count'].apply(lambda x: categorize_conflict(x, q1, q3))
features = df.drop(['GEOID', 'conflict_count', 'target'], axis=1)
features = pd.get_dummies(features, columns=['year'], prefix='year')
X = features
y = df['target']

In [5]:
# Preprocess data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train with gradient descent
# 'saga' is a variant of stochastic gradient descent

param_grid = {
    'estimator__C': np.logspace(-3, 2, 6),           # C: [0.001, 0.01, 0.1, 1, 10, 100]
    'estimator__l1_ratio': np.linspace(0, 1, 5)      # l1_ratio: [0, 0.25, 0.5, 0.75, 1]
}

base_model = LogisticRegression(
    solver='saga',
    penalty='elasticnet',
    max_iter=1000,
    random_state=42
)

ovr_model = OneVsRestClassifier(base_model)

grid = GridSearchCV(
    ovr_model,
    param_grid,
    cv=5,
    scoring='recall_weighted',  # or another multilabel-compatible metric
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)

# Results
print("Best C:", grid.best_params_['estimator__C'])
print("Best l1_ratio:", grid.best_params_['estimator__l1_ratio'])
print("Best score:", grid.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best C: 0.1
Best l1_ratio: 0.75
Best score: 0.9232691102328546


### Class imbalance

In [9]:
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report, recall_score

# Use the same preprocessing steps as before
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE for class imbalance
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Define the cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300, 500],        # Number of trees
    'max_depth': [None, 10, 20],       # Maximum depth of trees
    'min_samples_split': [2, 5, 10],   # Minimum samples required to split
    'min_samples_leaf': [1, 2, 4],     # Minimum samples required at leaf node
    'class_weight': ['balanced', None] # Adjust weights inversely proportional to class frequencies
}

# Initialize the Random Forest classifier
rf = RandomForestClassifier(random_state=42)

# Set up GridSearchCV with recall as the scoring metric
# You can use 'f1' or 'recall' depending on your priority
grid_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=50,                    # Try 50 random combinations
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='recall_weighted',     # Optimize for recall
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Fit the model
print("Training Random Forest with GridSearchCV...")
grid_search.fit(X_train_smote, y_train_smote)

# Get best model
best_rf = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")

# Predict on test set
y_pred_rf = best_rf.predict(X_test_scaled)

# Evaluate
print("\n--- Random Forest Model (Optimized) ---")
print(f"Accuracy: {best_rf.score(X_test_scaled, y_test):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_rf):.4f}")
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_rf)
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Conflict (0)', 'Conflict (1)'],
            yticklabels=['No Conflict (0)', 'Conflict (1)'])
plt.xlabel('Predicted Label', fontsize=14)
plt.ylabel('True Label', fontsize=14)
plt.title('Confusion Matrix - Random Forest', fontsize=16)
plt.tight_layout()
plt.show()

# Feature importance
feature_importance = best_rf.feature_importances_
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
})
importance_df = importance_df.sort_values('Importance', ascending=False).head(10)

# Plot feature importance
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Top 10 Features - Random Forest Importance', fontsize=16)
plt.xlabel('Importance Score', fontsize=14)
plt.ylabel('Feature', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.tight_layout()
plt.show()

Training Random Forest with GridSearchCV...
Fitting 5 folds for each of 50 candidates, totalling 250 fits


/Users/anfelipecb/Library/CloudStorage/Box-Box/MSCAPP/3rd Quarter/Machine Learning/Project/project-aeyzaguirre-phernandezpedraz-afcamachob/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'class_weight': None}

--- Random Forest Model (Optimized) ---
Accuracy: 0.8914


ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

### Now with multiclass target

In [ ]:
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report, recall_score

# Use the same preprocessing steps as before
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE for class imbalance
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Define the cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300, 500],        # Number of trees
    'max_depth': [None, 10, 20, 30],       # Maximum depth of trees
    'min_samples_split': [2, 5, 10,15],   # Minimum samples required to split
    'min_samples_leaf': [1, 2, 4, 6],     # Minimum samples required at leaf node
    'max_features': ['sqrt', 'log2', None, 0.8],
    'class_weight': ['balanced', None] # Adjust weights inversely proportional to class frequencies
}

# Initialize the Random Forest classifier
rf = RandomForestClassifier(random_state=42)

# Set up GridSearchCV with recall_weighted for multiclass
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=cv,
    scoring='recall_weighted',  # Changed to weighted recall for multiclass
    n_jobs=-1,                 
    verbose=1,
    n_iter=50, 
    random_state=42
)

# Fit the model
print("Training Random Forest with GridSearchCV...")
grid_search.fit(X_train_smote, y_train_smote)

# Get best model
best_rf = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")

# Predict on test set
y_pred_rf = best_rf.predict(X_test_scaled)

# Evaluate with appropriate multiclass metrics
print("\n--- Random Forest Model (Optimized) ---")
print(f"Accuracy: {best_rf.score(X_test_scaled, y_test):.4f}")
print(f"Recall (weighted): {recall_score(y_test, y_pred_rf, average='weighted'):.4f}")
print(f"F1 Score (weighted): {f1_score(y_test, y_pred_rf, average='weighted'):.4f}")

# Confusion matrix for multiclass
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_rf)
print(cm)

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# Visualize confusion matrix for multiclass
plt.figure(figsize=(10, 8))
class_labels = ['No Conflict (0)', 'Low (1)', 'Medium (2)', 'High (3)']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels,
            yticklabels=class_labels)
plt.xlabel('Predicted Label', fontsize=14)
plt.ylabel('True Label', fontsize=14)
plt.title('Confusion Matrix - Random Forest (Multiclass)', fontsize=16)
plt.tight_layout()
plt.show()

# Feature importance
feature_importance = best_rf.feature_importances_
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
})
importance_df = importance_df.sort_values('Importance', ascending=False).head(10)

# Plot feature importance
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Top 10 Features - Random Forest Importance', fontsize=16)
plt.xlabel('Importance Score', fontsize=14)
plt.ylabel('Feature', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.tight_layout()
plt.show()